# Avellaneda-Stoikov (AS) model

# State variables:

$s_t$: "reference price", mid, microprice, or fair value estimate

$q_t$: inventory (signed contracts)

$T$: time horizon, minutes unto we stop quoting, or event close

$\tau = T-t$: remaining time

$\sigma$: volatility of $s_t$ (in dollars per $\sqrt{\text{time}}$)

$\gamma$: risk aversion parameter (higher $\rightarrow$ more inventory-averse)

# Fill model (standard)

Assuming order arrivals hit our quotes with intensity that decays with distance $\delta$ from the reference price $s_t$:

\begin{align*}
    \lambda(\delta) = Ae^{k\delta}
\end{align*}

where:

$A$: baseline arrival rate

$k$: "liquidity slope", how quickly fills disappear as you move away

Note: This is not "true" in all markets, but it is a common and practical approximation; and crucially, $A, k$ are *calibratable* from our own data even in smaller venues.

# AS result: reservation price and optimal spread

## Reservation price (inventory-skewed center)

\begin{align*}
    r_t = s_t - q_t\gamma\sigma^2\tau
\end{align*}

If you are long ($q>0$), your "center" shifts downward; if short, upward. The magnitude is proportional to inventory, volatility, and remaining time.

## Optimal half-spread
One common closed-form expression under the exponential intensity model above is:
\begin{align*}
    \Delta_t = \underbrace{\frac{\gamma\sigma^2\tau}{2}}_{\text{inventory risk}} + \underbrace{\frac{1}{\gamma}\ln(1 + \frac{\gamma}{k})}_{\text{liquidity/adverse selection proxy}}
\end{align*}

So our quotes are:
\begin{align*}
    p_t^{\text{bid}} = r_t - \Delta_t,\quad p_t^{\text{ask}} = r_t + \Delta_t,
\end{align*}

defining our bids and asks.

# Justification in small markets

Kalshi-style markets, are thin, jumpy, and bounded $[0,1]$. The AS structure helps because it:

1. Explicitly widens when volatility rises (news/approaching resolution).
2. Automatically de-risks inventory as it grows, without us hand-tuning a "skew per contract" that becomes wrong when regimes change.
3. Separates two realities:
    * inventory risk term $\propto \sigma^2\tau$
    * fill/liquidity term $\propto \ln(1+\frac{\gamma}{k})$

In illiquid venues, "how far you can quote and stoll get hit" (our liquidity slope $k$) matters as much as spread.

# Practical adaptations for a bounded binary contract

## 1. Choose $s_t$ carefully (mid is often garbase in illiquid books!)
In thin markets, best bid/ask can be stale or empty. Common robust choices:
* Last trade price (with decay back toward book when fresh quotes appear)
* Microprice if both sides exist:
\begin{align*}
    s_t\approx \frac{a_t\cdot V_t^{\text{bid}} + b_t\cdot V_t^{\text{ask}}}{V_t^{\text{bid}} + V_t^{\text{ask}}}
\end{align*}
* A smooted estimator (EWMA) that resists single-lot noise.

## 2. Clamp and tick-round
After computing $p^{\text{bid}}, p^{\text{ask}}$, clamp to $[0.01, 0.99]$ and round to tick.

## 3. Horizon $\tau$ matter more in prediction markets
For stocks, $\tau$ is often "how long until we stop."

For prediction markets, a meaningful $\tau$ is also "time until resolution/until we plan to go flat," because holding inventory into resolution is qualitatively different risk.

# How to calibrate the parameters in practice (even with small data)

## Volatility $\sigma$

Estimate from reference-price returns:
* Use EWMA on $\Delta s_t$ or log-returns (bounded so plain returns are often fine):
\begin{align*}
    \sigma^2\leftarrow (1-\alpha)\sigma^2 + \alpha(\Delta s_t)^2
\end{align*}

## Liquidity slope $k$

Can estimate "fill probability vs distance" from our own quoting history.

Simple operational approach:
1. For each quote we place at distance $\delta$ from $s_t$, record whether it filled within a window $W$.
2. Fit an exponential decay model to fill frequency:
\begin{align*}
    \Pr(\text{fill in } W | \delta)\approx ce^{-k\delta}
\end{align*}

Even a rough $k$ is valuable; it tells us whether moving 1-2 ticks out kills our fills.

## Risk aversion $\gamma$

Pick $\gamma$ to eforce a desired "inventory shift" at a target inventory $q^*$.

For example, if we want the reservation-price shift to be about one tick $\Delta_{\text{tick}}$ when $|q|=q^*$:
\begin{align*}
    \Delta_{\text{tick}}\approx q^*\gamma\sigma^2\tau \Rightarrow \gamma\approx\frac{\Delta_{\text{tick}}}{q^*\sigma^2\tau}
\end{align*}

This is more pricipled than guessing "skew bps per contract."

# Crucial extension for prediction markets: "fair value" drift (information risk)

In prediction markets, there is often a real directional signal (polls, injury reports, weather, court decisions). A standard extension is to shift the reference price by a short-horizon alpha:
\begin{align*}
    s_t\leftarrow s_t + \alpha_t\tau
\end{align*}

This helps aovud being systematically adversely selected when informed traders hit your stale quotes.

In [3]:
import math
import time
from dataclasses import dataclass
from typing import Optional, Dict, Any

@dataclass
class Quote:
    bid: Optional[float]     # dollars, None => do not quote bid
    ask: Optional[float]     # dollars, None => do not quote ask
    ref: float               # s_t reference price
    reservation: float       # r_t
    half_spread: float       # Δ_t
    sigma: float             # instantaneous vol estimate (per sqrt(second))
    tau: float               # time-to-horizon in seconds
    position: int            # q_t

In [ ]:
class ASTrader:
    """
    Avellaneda-Stoikov-style market maker for bounded [0,1] binary prices.

    Core:
      r_t = s_t - q_t * gamma * sigma^2 * tau
      Delta_t = (gamma * sigma^2 * tau)/2  +  (1/gamma)*log(1 + gamma/k)   (with gamma -> theta limit handled)
      p_bid = r_t - Delta_t
      p_ask = r_t + Delta_t

    Practical bits:
      - s_t uses microprice if volumes provided; else mid.
      - sigma^2 estimated online via EWMA of (ds^2 / dt).
      - quotes are clamped to [min_price, max_price] and rounded to tick.
      - inventory bounds disable one side when at limits.
    """

    def __init__(
        self,
        tick: float = 0.01,
        min_price: float = 0.01,
        max_price: float = 0.99,
        max_position: int = 10,

        # AS parameters
        gamma: float = 0.05,              # risk aversion
        k: float = 25.0,                  # liquidity slope in exp intensity λ(δ)=A e^{-k δ}
        horizon_s: float = 300.0,         # T-t in seconds if you want a finite-horizon behavior

        # vol estimator
        ewma_alpha: float = 0.10,         # EWMA weight for new variance observations
        sigma2_floor: float = 1e-6,       # prevents collapse to 0; units: price^2 / sec
        sigma2_cap: float = 1.0,          # sanity cap; units: price^2 / sec

        # state init
        init_ref: float = 0.50,
    ):
        # market microstructure
        self.tick = float(tick)
        self.min_price = float(min_price)
        self.max_price = float(max_price)

        # inventory/risk
        self.max_position = int(max_position)
        self.gamma = float(gamma)
        self.k = float(k)
        self.horizon_s = float(horizon_s)

        # volatility estimate (sigma^2 in price^2 / sec)
        self.ewma_alpha = float(ewma_alpha)
        self.sigma2_floor = float(sigma2_floor)
        self.sigma2_cap = float(sigma2_cap)
        self.sigma2 = max(self.sigma2_floor, 0.0)

        # trading state (simple cash+inventory accounting; no fees)
        self.position = 0
        self.cash = 0.0  # buy decreases cash, sell increases cash

        # reference price tracking
        self.last_ref = float(init_ref)
        self.last_ts = None  # epoch seconds
        self.start_ts = None

    # ---------- reference price ----------
    def _compute_st_ref_price(self, ask: float, bid: float, V_ask: float, V_bid: float) -> float:
        # microprice
        return (ask * V_bid + bid * V_ask) / (V_ask + V_bid)

    def _compute_reference(self, bid: Optional[float], ask: Optional[float],
                           V_bid: Optional[float], V_ask: Optional[float]) -> float:
        # Prefer microprice when both sides + volumes exist; else mid; else fallback last_ref.
        if bid is None or ask is None or not (ask > bid > 0):
            return self.last_ref

        if V_bid is not None and V_ask is not None and (V_bid + V_ask) > 0:
            return self._compute_st_ref_price(ask, bid, V_ask, V_bid)

        return 0.5 * (bid + ask)

    # ---------- volatility (EWMA on instantaneous variance) ----------
    def _update_sigma2(self, ref: float, ts: float) -> None:
        if self.last_ts is None:
            self.last_ts = ts
            self.last_ref = ref
            self.sigma2 = max(self.sigma2_floor, self.sigma2)
            return

        dt = ts - self.last_ts
        if dt <= 0:
            # no time elapsed; keep prior estimate
            self.last_ref = ref
            return

        ds = ref - self.last_ref
        inst_sigma2 = (ds * ds) / dt  # diffusion-consistent estimate of sigma^2
        inst_sigma2 = min(max(inst_sigma2, self.sigma2_floor), self.sigma2_cap)

        a = self.ewma_alpha
        self.sigma2 = (1.0 - a) * self.sigma2 + a * inst_sigma2
        self.sigma2 = min(max(self.sigma2, self.sigma2_floor), self.sigma2_cap)

        self.last_ts = ts
        self.last_ref = ref

    # ---------- AS formulas ----------
    def _tau(self, ts: float) -> float:
        if self.start_ts is None:
            self.start_ts = ts
        elapsed = ts - self.start_ts
        return max(0.0, self.horizon_s - elapsed)

    def _reservation_price(self, s: float, q: int, sigma2: float, tau: float) -> float:
        return s - (q * self.gamma * sigma2 * tau)

    def _liquidity_term(self) -> float:
        """
        (1/gamma) * log(1 + gamma/k), with γ→0 limit = 1/k.
        """
        if self.k <= 0:
            # pathological; avoid blow-up
            return 0.0
        g = self.gamma
        if abs(g) < 1e-9:
            return 1.0 / self.k
        return (1.0 / g) * math.log(1.0 + (g / self.k))

    def _half_spread(self, sigma2: float, tau: float) -> float:
        inv_risk = 0.5 * self.gamma * sigma2 * tau
        liq = self._liquidity_term()
        return max(0.0, inv_risk + liq)

    # ---------- rounding/clamping ----------
    def _clamp(self, p: float) -> float:
        return max(self.min_price, min(self.max_price, p))

    def _floor_to_tick(self, p: float) -> float:
        # good for bids
        t = self.tick
        return math.floor(p / t + 1e-12) * t

    def _ceil_to_tick(self, p: float) -> float:
        # good for asks
        t = self.tick
        return math.ceil(p / t - 1e-12) * t

    # ---------- public API ----------
    def update_market(
        self,
        bid: Optional[float],
        ask: Optional[float],
        V_bid: Optional[float] = None,
        V_ask: Optional[float] = None,
        ts: Optional[float] = None,
    ) -> float:
        """
        Feed the latest top-of-book (and optionally sizes) into the model.
        Returns the updated reference price s_t.
        """
        now = time.time() if ts is None else float(ts)
        s = self._compute_reference(bid, ask, V_bid, V_ask)
        self._update_sigma2(s, now)
        return s

    def compute_quotes(self, ts: Optional[float] = None) -> Quote:
        """
        Compute AS quotes using the current state.
        You should call update_market(...) before this each loop tick.
        """
        now = time.time() if ts is None else float(ts)

        s = self.last_ref
        tau = self._tau(now)
        sigma2 = self.sigma2
        sigma = math.sqrt(max(sigma2, 0.0))

        q = self.position
        r = self._reservation_price(s, q, sigma2, tau)
        d = self._half_spread(sigma2, tau)

        raw_bid = r - d
        raw_ask = r + d

        # Clamp to valid probability interval
        raw_bid = self._clamp(raw_bid)
        raw_ask = self._clamp(raw_ask)

        # Round to tick: bid floors, ask ceils
        bid = self._floor_to_tick(raw_bid)
        ask = self._ceil_to_tick(raw_ask)

        # Enforce at least one tick of spread
        if ask <= bid:
            mid_tick = self._clamp(0.5 * (bid + ask))
            bid = self._floor_to_tick(max(self.min_price, mid_tick - self.tick))
            ask = self._ceil_to_tick(min(self.max_price, mid_tick + self.tick))
            if ask <= bid:
                # last-resort: widen by one tick from clamp edges
                bid = max(self.min_price, min(self.max_price - self.tick, bid))
                ask = max(bid + self.tick, min(self.max_price, ask))

        # Inventory constraints: disable one side if at limits
        if self.position >= self.max_position:
            bid = None
        if self.position <= -self.max_position:
            ask = None

        return Quote(
            bid=bid,
            ask=ask,
            ref=s,
            reservation=r,
            half_spread=d,
            sigma=sigma,
            tau=tau,
            position=q,
        )

    def on_fill(self, action: str, price: float, quantity: int) -> Dict[str, Any]:
        """
        Update inventory and cash upon fills.
        action: "buy" or "sell"
        price: dollars in [0,1]
        quantity: contracts
        Returns a mark-to-market snapshot at current reference price.
        """
        if quantity <= 0:
            return {"position": self.position, "cash": self.cash, "mtm": self.mtm_pnl()}

        p = float(price)
        q = int(quantity)

        if action == "buy":
            self.position += q
            self.cash -= q * p
        elif action == "sell":
            self.position -= q
            self.cash += q * p
        else:
            raise ValueError(f"action must be 'buy' or 'sell', got {action!r}")

        return {"position": self.position, "cash": self.cash, "mtm": self.mtm_pnl()}

    def mtm_pnl(self, ref: Optional[float] = None) -> float:
        """
        Mark-to-market P&L using cash + q*s_t.
        (Fees ignored; for Kalshi you likely want to subtract fees per fill.)
        """
        s = self.last_ref if ref is None else float(ref)
        return self.cash + self.position * s

In [ ]:
mm = ASTrader(
    gamma=0.05,
    k=25.0,
    horizon_s=300,
    max_position=10
)

